In [ ]:
@title Dependency installation
!pip install -q --upgrade umap-learn numba
!pip install -q scikit-learn joblib matplotlib torch numpy

ERROR: Operation cancelled by user


KeyboardInterrupt: 

In [ ]:
# @title train_gru_param.py

import os
from pathlib import Path
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, random_split

current_script_path = Path(__file__).resolve()
parent_dir = current_script_path.parent
array_folder= parent_dir/ "model_arrays"
latents_folder= current_script_path/"latents_folder"
PROJECT_DIR = array_folder
TRAIN_NPY   = os.path.join(PROJECT_DIR, "train_array.npy")

MODEL       = "xy"
LATENT_DIM  = 8
HIDDEN_DIM  = 128
NUM_LAYERS  = 1
INPUT_DIM   = 2

EPOCHS        = 15
BATCH_SIZE    = 256
LR            = 1e-3
VAL_SPLIT     = 0.10
SEED          = 42
KL_WEIGHT_MAX = 1e-3
KL_WARMUP_EP  = 20
SS_WARMUP_EP  = 10
TF_PROB_MIN   = 0.3
GRAD_CLIP     = 1.0

OUT_DIR = os.path.join(PROJECT_DIR, f"{MODEL}_parameters_latentDim{LATENT_DIM}")
os.makedirs(OUT_DIR, exist_ok=True)

torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Μοντέλο με πέρασμα dxdy ή xy για σύγκριση απόδοσης
def build_features(positions: np.ndarray, model: str) -> np.ndarray:
    if model == "xy":
        return positions.astype(np.float32)
    if model == "dxdy":
        features = np.zeros_like(positions, dtype=np.float32)
        features[:, 1:, 0] = positions[:, 1:, 0] - positions[:, :-1, 0]
        features[:, 1:, 1] = positions[:, 1:, 1] - positions[:, :-1, 1]
        return features
    raise ValueError("MODEL must be 'xy' or 'dxdy'")


class TrajectoryDataset(Dataset):
    def __init__(self, features: np.ndarray):
        self.features = features

    def __len__(self):
        return len(self.features)

    def __getitem__(self, i):
        return torch.from_numpy(self.features[i])


class BiGRUEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim, num_layers):
        super().__init__()
        self.gru = nn.GRU(
            input_dim, hidden_dim, num_layers,
            batch_first=True, bidirectional=True,
            dropout=0.1 if num_layers > 1 else 0.0,
        )
        self.fc_mu = nn.Linear(2 * hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(2 * hidden_dim, latent_dim)

    def forward(self, x):
        out, _ = self.gru(x)
        pooled = out.mean(dim=1)
        mu = self.fc_mu(pooled)
        logvar = self.fc_logvar(pooled).clamp(-8, 8)
        return mu, logvar


class GRUDecoder(nn.Module):
    def __init__(self, latent_dim, hidden_dim, output_dim, seq_len, num_layers):
        super().__init__()
        self.seq_len = seq_len
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.fc_init = nn.Linear(latent_dim, num_layers * hidden_dim)
        self.gru = nn.GRU(
            output_dim, hidden_dim, num_layers,
            batch_first=True,
            dropout=0.1 if num_layers > 1 else 0.0,
        )
        self.out = nn.Linear(hidden_dim, output_dim)

    def forward(self, z, target_reversed=None, tf_prob=1.0):
        B = z.size(0)
        h = torch.tanh(self.fc_init(z)).view(self.num_layers, B, self.hidden_dim).contiguous()
        prev = torch.zeros(B, 1, self.out.out_features, device=z.device)
        outputs = []
        for t in range(self.seq_len):
            o, h = self.gru(prev, h)
            step = self.out(o)
            outputs.append(step)
            if target_reversed is not None and t < self.seq_len - 1 and tf_prob > 0:
                use_tf = torch.rand(B, 1, 1, device=z.device) < tf_prob
                gt_next = target_reversed[:, t:t+1, :]
                prev = torch.where(use_tf, gt_next, step)
            else:
                prev = step
        return torch.cat(outputs, dim=1)


class TrajectoryVAE(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim, num_layers, seq_len):
        super().__init__()
        self.encoder = BiGRUEncoder(input_dim, hidden_dim, latent_dim, num_layers)
        self.decoder = GRUDecoder(latent_dim, hidden_dim, input_dim, seq_len, num_layers)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + std * torch.randn_like(std)

    def forward(self, x, tf_prob=1.0):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar) if self.training else mu
        x_rev = torch.flip(x, dims=[1])
        x_hat_rev = self.decoder(z, target_reversed=x_rev, tf_prob=tf_prob)
        x_hat = torch.flip(x_hat_rev, dims=[1])
        return x_hat, mu, logvar, z


def train():
    positions_raw = np.load(TRAIN_NPY).astype(np.float32)
    _, T, _ = positions_raw.shape
    features_all = build_features(positions_raw, MODEL)

    dataset = TrajectoryDataset(features_all)
    val_size = max(1, int(len(dataset) * VAL_SPLIT))
    train_size = len(dataset) - val_size
    gen = torch.Generator().manual_seed(SEED)
    train_ds, val_ds = random_split(dataset, [train_size, val_size], generator=gen)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

    model = TrajectoryVAE(INPUT_DIM, HIDDEN_DIM, LATENT_DIM, NUM_LAYERS, T).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
    recon_loss_fn = nn.SmoothL1Loss(reduction="mean")

    best_val = float("inf")
    best_path = os.path.join(OUT_DIR, f"gru_autoencoder_{MODEL}_latent{LATENT_DIM}_best.pt")

    for epoch in range(1, EPOCHS + 1):
        kl_weight = KL_WEIGHT_MAX * min(1.0, epoch / KL_WARMUP_EP)
        tf_prob = max(TF_PROB_MIN, 1.0 - (1.0 - TF_PROB_MIN) * (epoch - 1) / max(1, SS_WARMUP_EP))

        model.train()
        t_recon = t_kl = n_seen = 0
        for seqs in train_loader:
            seqs = seqs.to(device, non_blocking=True)
            optimizer.zero_grad()
            x_hat, mu, logvar, _ = model(seqs, tf_prob=tf_prob)
            recon = recon_loss_fn(x_hat, seqs)
            kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
            loss = recon + kl_weight * kl
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            bs = seqs.size(0)
            t_recon += recon.item() * bs
            t_kl += kl.item() * bs
            n_seen += bs
        t_recon /= n_seen
        t_kl /= n_seen

        model.eval()
        v_recon = n_val = 0
        with torch.no_grad():
            for seqs in val_loader:
                seqs = seqs.to(device, non_blocking=True)
                x_hat, _, _, _ = model(seqs, tf_prob=0.0)
                v_recon += recon_loss_fn(x_hat, seqs).item() * seqs.size(0)
                n_val += seqs.size(0)
        v_recon /= n_val

        scheduler.step()

        if epoch == 1 or epoch % 5 == 0:
            print(f"Ep {epoch:3d}/{EPOCHS} | recon_t={t_recon:.5f} recon_v={v_recon:.5f} kl={t_kl:.4f} tf={tf_prob:.2f} klw={kl_weight:.1e} lr={scheduler.get_last_lr()[0]:.2e}")

        if v_recon < best_val:
            best_val = v_recon
            torch.save(model.state_dict(), best_path)

    torch.save(model.encoder.state_dict(), os.path.join(OUT_DIR, f"gru_encoder_{MODEL}_latent{LATENT_DIM}.pt"))
    torch.save(model.state_dict(), os.path.join(OUT_DIR, f"gru_autoencoder_{MODEL}_latent{LATENT_DIM}.pt"))

    state = torch.load(best_path, map_location=device)
    model.load_state_dict(state)
    model.eval()

    all_ds = TrajectoryDataset(features_all)
    all_loader = DataLoader(all_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    embs = []
    with torch.no_grad():
        for seqs in all_loader:
            mu, _ = model.encoder(seqs.to(device))
            embs.append(mu.cpu().numpy())
    embs = np.concatenate(embs, axis=0)
    np.save(os.path.join(OUT_DIR, f"trajectory_embeddings_{MODEL}_latent{LATENT_DIM}.npy"), embs)

    print(f"Best val recon : {best_val:.6f}")
    print(f"Embeddings shape: {embs.shape}")


if __name__ == "__main__":
    train()



In [ ]:
# @title umap based clustering & manifold Creation { display-mode: "form" }



import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score, normalized_mutual_info_score
import joblib

# Config
MODEL      = "xy"
LATENT_DIM = 16
N_CLUSTERS = 6
SCATTER_N  = 20_000
TRAJ_N     = 20
SIL_N      = 2_000
N_PER_ROW  = 4

PROJECT_DIR = "."
#PROJECT_DIR = "/content/drive/My Drive/Colab Notebooks/..." #για τρέξιμο σε colab
MODEL_DIR   = os.path.join(PROJECT_DIR, f"{MODEL}_parameters_latentDim{LATENT_DIM}")
EMB_PATH    = os.path.join(MODEL_DIR,   f"trajectory_embeddings_{MODEL}_latent{LATENT_DIM}.npy")
LABELS_PATH = os.path.join(PROJECT_DIR, "train_array_labels.npy")
TRAJ_PATH   = os.path.join(PROJECT_DIR, "train_array.npy")

OBJECT_MAP = {0:"vehicle", 1:"pedestrian", 2:"construction", 3:"riderless_bicycle",
              4:"bus", 5:"background", 6:"static", 7:"cyclist", 8:"motorcyclist", 9:"unknown"}
DIR_MAP    = {0:"straight", 1:"u_turn", 2:"left_turn", 3:"right_turn", 4:"unspecified"}
ACCEL_MAP  = {0:"constant", 1:"decelerate", 2:"accelerate"}
SPEED_MAP  = {i: f"speed_{i}" for i in range(5)}

# 1. φόρτωμα δεδομένων
Z         = np.load(EMB_PATH).astype(np.float32)
raw_lab   = np.load(LABELS_PATH).astype(np.int64)
positions = np.load(TRAJ_PATH).astype(np.float32)
N         = len(Z)

labels = np.full((N, 4), -1, dtype=np.int64)
labels[:min(N, len(raw_lab))] = raw_lab[:min(N, len(raw_lab))]
valid = np.all(labels >= 0, axis=1)

dpos           = np.diff(positions, axis=1)
step_speed     = np.linalg.norm(dpos, axis=2)
avg_speed      = step_speed.mean(axis=1)
log_step_speed = np.log(step_speed + 1e-6)

# 2. λογαριθμικά δεδομένα για umap
Z_log      = np.sign(Z) * np.log(np.abs(Z) + 1.0)
scaler_log = StandardScaler().fit(Z_log)
Zs_log     = scaler_log.transform(Z_log)

# 3. Umap
import umap as umap_mod
reducer = umap_mod.UMAP(n_components=2, n_neighbors=30, min_dist=0.1, random_state=42)
Z2_all  = reducer.fit_transform(Zs_log)

# 4. KMEANS πάνω στο umap
scaler_cluster = StandardScaler().fit(Z2_all)
Zs_cluster     = scaler_cluster.transform(Z2_all)
kmeans         = KMeans(n_clusters=N_CLUSTERS, n_init=10, random_state=42)
cluster        = kmeans.fit_predict(Zs_cluster)

# 5. Δειγματοληψία
rng  = np.random.default_rng(42)
sidx = rng.choice(N, size=min(SCATTER_N, N), replace=False)
Z2   = Z2_all[sidx]
xl, yl = "UMAP-1", "UMAP-2"

# 6. Γραφήματα
def _scatter_cat(ax, title, vals, label_map):
    v    = vals[sidx]
    mask = v >= 0
    ax.scatter(Z2[~mask, 0], Z2[~mask, 1], c="#d0d0d0", s=3, alpha=0.15, rasterized=True)
    classes = sorted(np.unique(v[mask]))
    colors  = plt.cm.tab10(np.linspace(0, 1, max(len(classes), 1)))
    all_idx = np.where(mask)[0]
    shuf    = rng.permutation(len(all_idx))
    s_idx   = all_idx[shuf]; s_vals = v[s_idx]; s_Z2 = Z2[s_idx]
    for uid, col in zip(classes, colors):
        m = s_vals == uid
        ax.scatter(s_Z2[m, 0], s_Z2[m, 1], s=4, alpha=0.55,
                   color=col, rasterized=True, label=label_map.get(int(uid), str(uid)))
    ax.legend(markerscale=3, fontsize=7, loc="best", framealpha=0.6)
    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.set_xlabel(xl, fontsize=8); ax.set_ylabel(yl, fontsize=8)
    ax.tick_params(labelsize=7)

def _scatter_cont(ax, title, vals, cmap="viridis", cbar_lbl="", log_scale=False):
    v = vals[sidx]
    if log_scale:
        v = np.log(v + 1e-6)
    sc = ax.scatter(Z2[:, 0], Z2[:, 1], c=v, cmap=cmap, s=4, alpha=0.6,
                    vmin=np.percentile(v, 1), vmax=np.percentile(v, 99), rasterized=True)
    plt.colorbar(sc, ax=ax, pad=0.02).set_label(cbar_lbl, fontsize=8)
    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.set_xlabel(xl, fontsize=8); ax.set_ylabel(yl, fontsize=8)
    ax.tick_params(labelsize=7)


fig1, axes1 = plt.subplots(1, 7, figsize=(42, 5.5))
fig1.suptitle(
    f"Train — Latent Space  [XY | latent={LATENT_DIM}]  —  {N_CLUSTERS} clusters"
    f"  (UMAP→KMeans on log-embedding)",
    fontsize=12, fontweight="bold", y=1.01)

_scatter_cat(axes1[0],  f"KMeans  k={N_CLUSTERS}", cluster,      {i: f"C{i}" for i in range(N_CLUSTERS)})
_scatter_cat(axes1[1],  "Object type",              labels[:, 0], OBJECT_MAP)
_scatter_cat(axes1[2],  "Direction",                labels[:, 1], DIR_MAP)
_scatter_cat(axes1[3],  "Acceleration",             labels[:, 2], ACCEL_MAP)
_scatter_cat(axes1[4],  "Speed label (discrete)",   labels[:, 3], SPEED_MAP)
_scatter_cont(axes1[5], "Avg speed (linear)",       avg_speed,    cbar_lbl="avg ‖Δpos‖ / step")
_scatter_cont(axes1[6], "Avg speed (log)",          avg_speed,    cbar_lbl="log(avg ‖Δpos‖ + ε)",
              cmap="plasma", log_scale=True)

plt.tight_layout()
plt.show()


# 7. Στατιστικά
sil_idx = rng.choice(N, size=min(SIL_N, N), replace=False)
sil     = silhouette_score(Zs_cluster[sil_idx], cluster[sil_idx])
print(f"\nModel : XY | latent={LATENT_DIM} | k={N_CLUSTERS} | "
      f"Silhouette={sil:.4f} | clustering=UMAP→KMeans")
print(f"  {'Label':<15} {'ARI':>8} {'NMI':>8}")
for name, col, mp in [("object_type",0,OBJECT_MAP),("direction",1,DIR_MAP),
                       ("acceleration",2,ACCEL_MAP),("speed",3,SPEED_MAP)]:
    vm  = valid & (labels[:, col] >= 0)
    ari = adjusted_rand_score(labels[vm, col], cluster[vm])
    nmi = normalized_mutual_info_score(labels[vm, col], cluster[vm])
    print(f"  {name:<15} {ari:>8.4f} {nmi:>8.4f}")

# 8. Αποθήκευση για αργότερα του
os.makedirs(MODEL_DIR, exist_ok=True)
joblib.dump(scaler_log,     os.path.join(MODEL_DIR, "scaler_log.pkl"))
joblib.dump(reducer,        os.path.join(MODEL_DIR, "umap_reducer.pkl"))
joblib.dump(scaler_cluster, os.path.join(MODEL_DIR, "scaler_cluster.pkl"))
joblib.dump(kmeans,         os.path.join(MODEL_DIR, "kmeans.pkl"))
np.save(os.path.join(MODEL_DIR, "cluster_labels_train.npy"), cluster)


In [ ]:
# @title test points model pass and label cross-check { display-mode: "form" }



import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import joblib
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

# Config
MODEL       = "xy"
LATENT_DIM  = 8
HIDDEN_DIM  = 128
NUM_LAYERS  = 1
INPUT_DIM   = 2
BATCH_SIZE  = 256
N_CLUSTERS  = 6
SCATTER_N   = 20_000
TRAJ_N      = 20
N_PER_ROW   = 4

#PROJECT_DIR = "/content/drive/My Drive/Colab Notebooks/project/encoderNPY" #για τρέξιμο σε colab
PROJECT_DIR = "."

MODEL_DIR   = os.path.join(PROJECT_DIR, f"{MODEL}_parameters_latentDim{LATENT_DIM}")

TEST_NPY        = os.path.join(PROJECT_DIR, "noisy_test_array_encoder_001.npy")
TEST_LABELS_NPY = os.path.join(PROJECT_DIR, "test_array_labels.npy")
ENCODER_PT      = os.path.join(MODEL_DIR,  f"gru_encoder_{MODEL}_latent{LATENT_DIM}.pt")

SCALER_LOG_PATH   = os.path.join(MODEL_DIR, "scaler_log.pkl")
REDUCER_PATH      = os.path.join(MODEL_DIR, "umap_reducer.pkl")
SCALER_CLUST_PATH = os.path.join(MODEL_DIR, "scaler_cluster.pkl")
KMEANS_PATH       = os.path.join(MODEL_DIR, "kmeans.pkl")

OUT_EMB_PATH     = os.path.join(MODEL_DIR, f"trajectory_embeddings_{MODEL}_latent{LATENT_DIM}_test.npy")
OUT_EMB_LOG_PATH = os.path.join(MODEL_DIR, f"trajectory_embeddings_log_{MODEL}_latent{LATENT_DIM}_test.npy")
OUT_UMAP_PATH    = os.path.join(MODEL_DIR, f"umap_2d_{MODEL}_latent{LATENT_DIM}_test.npy")
OUT_LABELS_PATH  = os.path.join(MODEL_DIR, "cluster_labels_test.npy")

OBJECT_MAP = {0:"vehicle", 1:"pedestrian", 2:"construction", 3:"riderless_bicycle",
              4:"bus", 5:"background", 6:"static", 7:"cyclist", 8:"motorcyclist", 9:"unknown"}
DIR_MAP    = {0:"straight", 1:"u_turn", 2:"left_turn", 3:"right_turn", 4:"unspecified"}
ACCEL_MAP  = {0:"constant", 1:"decelerate", 2:"accelerate"}
SPEED_MAP  = {i: f"speed_{i}" for i in range(5)}

# Διαφορετικά χρώματα για τα plots
PALETTE_CLUSTER = ["#e6194b","#3cb44b","#4363d8","#f58231","#911eb4",
                   "#42d4f4","#f032e6","#bfef45","#fabed4","#469990"]
PALETTE_OBJECT  = ["#e6194b","#3cb44b","#4363d8","#f58231","#911eb4",
                   "#42d4f4","#f032e6","#bfef45","#fabed4","#469990"]
PALETTE_DIR     = ["#e6194b","#4363d8","#3cb44b","#f58231","#911eb4"]
PALETTE_ACCEL   = ["#4363d8","#e6194b","#3cb44b"]
PALETTE_SPEED   = ["#4363d8","#3cb44b","#f5c518","#f58231","#e6194b"]
SPEED_HIGHLIGHT_COLORS = {
    0: "#4363d8",
    1: "#3cb44b",
    2: "#f5c518",
    3: "#f58231",
    4: "#e6194b",
}

def _get_palette(label_map, classes):
    if label_map is SPEED_MAP:   pal = PALETTE_SPEED
    elif label_map is ACCEL_MAP: pal = PALETTE_ACCEL
    elif label_map is DIR_MAP:   pal = PALETTE_DIR
    elif label_map is OBJECT_MAP:pal = PALETTE_OBJECT
    else:                        pal = PALETTE_CLUSTER
    return [pal[int(c) % len(pal)] for c in classes]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Κωδικοποιητής
class BiGRUEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim, num_layers):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, num_layers,
                          batch_first=True, bidirectional=True,
                          dropout=0.1 if num_layers > 1 else 0.0)
        self.fc_mu     = nn.Linear(2 * hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(2 * hidden_dim, latent_dim)

    def forward(self, x):
        out, _ = self.gru(x)
        pooled = out.mean(dim=1)
        mu     = self.fc_mu(pooled)
        logvar = self.fc_logvar(pooled).clamp(-8, 8)
        return mu, logvar

class TrajectoryDataset(Dataset):
    def __init__(self, data): self.data = data
    def __len__(self):        return len(self.data)
    def __getitem__(self, i): return torch.from_numpy(self.data[i])

# Φορτωμα τροχιών
positions_test = np.load(TEST_NPY).astype(np.float32)
features_test  = positions_test
N_test_raw   = len(positions_test)
raw_lab_test = np.load(TEST_LABELS_NPY).astype(np.int64)
labels_test  = np.full((N_test_raw, 4), -1, dtype=np.int64)
labels_test[:min(N_test_raw, len(raw_lab_test))] = raw_lab_test[:min(N_test_raw, len(raw_lab_test))]

# Τροχιές -> Z
encoder = BiGRUEncoder(INPUT_DIM, HIDDEN_DIM, LATENT_DIM, NUM_LAYERS).to(device)
encoder.load_state_dict(torch.load(ENCODER_PT, map_location=device))
encoder.eval()
loader = DataLoader(TrajectoryDataset(features_test),
                    batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
embs = []
with torch.no_grad():
    for seqs in loader:
        mu, _ = encoder(seqs.to(device))
        embs.append(mu.cpu().numpy())
Z_test = np.concatenate(embs, axis=0).astype(np.float32)

# Φορτωμα clusters από το train
scaler_log     = joblib.load(SCALER_LOG_PATH)
reducer        = joblib.load(REDUCER_PATH)
scaler_cluster = joblib.load(SCALER_CLUST_PATH)
kmeans         = joblib.load(KMEANS_PATH)


# Λογαριθμική αναπαράσταση
Z_test_log      = np.sign(Z_test) * np.log(np.abs(Z_test) + 1.0)
Zs_test_log     = scaler_log.transform(Z_test_log)

Z2_test         = reducer.transform(Zs_test_log)
Zs_test_cluster = scaler_cluster.transform(Z2_test)
cluster_labels  = kmeans.predict(Zs_test_cluster)

# Στατιστικά
N_test  = len(cluster_labels)
unique, counts = np.unique(cluster_labels, return_counts=True)
print(f"\\n{'-'*52}")
print(f"KMeans cluster assignment for {N_test:,} test trajectories")
print(f"{'-'*52}")
for c, cnt in zip(unique, counts):
    print(f"  C{int(c):<9}  {cnt:>8,}  {cnt/N_test*100:>5.1f}%")
print(f"{'-'*52}")

# Αποθήκευση αναπαραστάσεων
np.save(OUT_EMB_PATH,     Z_test)
np.save(OUT_EMB_LOG_PATH, Z_test_log)
np.save(OUT_UMAP_PATH,    Z2_test)
np.save(OUT_LABELS_PATH,  cluster_labels)

# Διαγράμματα
dpos_test           = np.diff(positions_test, axis=1)
step_speed_test     = np.linalg.norm(dpos_test, axis=2)
avg_speed_test      = step_speed_test.mean(axis=1)
log_step_speed_test = np.log(step_speed_test + 1e-6)

rng  = np.random.default_rng(0)
sidx = rng.choice(N_test, size=min(SCATTER_N, N_test), replace=False)
Z2t  = Z2_test[sidx]
xl, yl = "UMAP-1", "UMAP-2"

def _scatter_cat(ax, title, vals, label_map):
    v    = vals[sidx]
    mask = v >= 0
    ax.scatter(Z2t[~mask, 0], Z2t[~mask, 1], c="#d0d0d0", s=3, alpha=0.15, rasterized=True)
    classes = sorted(np.unique(v[mask])) if mask.any() else []
    colors  = _get_palette(label_map, classes)
    if mask.any():
        all_idx = np.where(mask)[0]
        shuf    = rng.permutation(len(all_idx))
        s_idx   = all_idx[shuf]; s_vals = v[s_idx]; s_Z2 = Z2t[s_idx]
        for uid, col in zip(classes, colors):
            m = s_vals == uid
            ax.scatter(s_Z2[m, 0], s_Z2[m, 1], s=4, alpha=0.55,
                       color=col, rasterized=True, label=label_map.get(int(uid), str(uid)))
    ax.legend(markerscale=3, fontsize=7, loc="best", framealpha=0.6)
    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.set_xlabel(xl, fontsize=8); ax.set_ylabel(yl, fontsize=8)
    ax.tick_params(labelsize=7)

def _scatter_cont(ax, title, vals, cmap="viridis", cbar_lbl="", log_scale=False):
    v = vals[sidx]
    if log_scale: v = np.log(v + 1e-6)
    sc = ax.scatter(Z2t[:, 0], Z2t[:, 1], c=v, cmap=cmap, s=4, alpha=0.6,
                    vmin=np.percentile(v, 1), vmax=np.percentile(v, 99), rasterized=True)
    plt.colorbar(sc, ax=ax, pad=0.02).set_label(cbar_lbl, fontsize=8)
    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.set_xlabel(xl, fontsize=8); ax.set_ylabel(yl, fontsize=8)
    ax.tick_params(labelsize=7)

fig1, axes1 = plt.subplots(1, 7, figsize=(42, 5.5))
fig1.suptitle(
    f"Test Set — Latent Space  [XY | latent={LATENT_DIM}]  —  {N_CLUSTERS} clusters"
    f"  (UMAP manifold from training | KMeans labels)",
    fontsize=12, fontweight="bold", y=1.01)

_scatter_cat(axes1[0],  f"KMeans  k={N_CLUSTERS}",  cluster_labels,    {i: f"C{i}" for i in range(N_CLUSTERS)})
_scatter_cat(axes1[1],  "Object type",               labels_test[:, 0], OBJECT_MAP)
_scatter_cat(axes1[2],  "Direction",                 labels_test[:, 1], DIR_MAP)
_scatter_cat(axes1[3],  "Acceleration",              labels_test[:, 2], ACCEL_MAP)
_scatter_cat(axes1[4],  "Speed label (discrete)",    labels_test[:, 3], SPEED_MAP)
_scatter_cont(axes1[5], "Avg speed (linear)",        avg_speed_test,    cbar_lbl="avg ‖Δpos‖ / step")
_scatter_cont(axes1[6], "Avg speed (log)",           avg_speed_test,    cbar_lbl="log(avg ‖Δpos‖ + ε)",
              cmap="plasma", log_scale=True)

plt.tight_layout()
plt.show()

speed_vals = labels_test[sidx, 3]
n_speed    = len(SPEED_MAP)

fig2, axes2 = plt.subplots(1, n_speed, figsize=(6 * n_speed, 5.5))
fig2.suptitle(
    f"Test Set — Speed Label Spotlight  [XY | latent={LATENT_DIM}]\\n"
    f"Each panel highlights one speed bucket; all other points are light grey",
    fontsize=12, fontweight="bold", y=1.02)

for spd_id in range(n_speed):
    ax    = axes2[spd_id]
    color = SPEED_HIGHLIGHT_COLORS[spd_id]
    label = SPEED_MAP[spd_id]

    is_target = speed_vals == spd_id
    is_other  = (speed_vals >= 0) & (~is_target)
    is_nodata = speed_vals < 0

    ax.scatter(Z2t[is_nodata, 0], Z2t[is_nodata, 1],
               c="#e0e0e0", s=3, alpha=0.10, rasterized=True)

    ax.scatter(Z2t[is_other, 0], Z2t[is_other, 1],
               c="#c8c8c8", s=3, alpha=0.25, rasterized=True)

    n_highlight = is_target.sum()
    ax.scatter(Z2t[is_target, 0], Z2t[is_target, 1],
               c=color, s=8, alpha=0.75, rasterized=True,
               label=f"{label}  (n={n_highlight:,})")

    ax.legend(markerscale=3, fontsize=9, loc="best", framealpha=0.7)
    ax.set_title(f"speed_{spd_id}", fontsize=11, fontweight="bold", color=color)
    ax.set_xlabel(xl, fontsize=8); ax.set_ylabel(yl, fontsize=8)
    ax.tick_params(labelsize=7)

plt.tight_layout()
plt.show()

